In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/layoffs.csv")

df.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
0,Pentera,Boston,40.0,4/27/2026,0.08,Security,https://www.calcalistech.com/ctechnews/article...,Series D,249.0,United States,4/30/2026
1,SuperOps,"Chennai, Non-U.S.",60.0,4/24/2026,0.30,Other,https://inc42.com/buzz/superops-lays-off-30-st...,Series C,54.0,India,4/30/2026
2,Acko,"Mumbai, Non-U.S.",60.0,4/20/2026,0.05,Finance,https://inc42.com/buzz/acko-cuts-5-workforce-i...,Unknown,143.0,India,4/20/2026
3,Meta,SF Bay Area,8000.0,4/17/2026,0.10,Consumer,https://www.reuters.com/world/meta-targets-may...,Post-IPO,26000.0,United States,4/20/2026
4,Shutterfly,"Haifa, Non-U.S.",80.0,4/16/2026,NaN,Manufacturing,https://www.calcalistech.com/ctechnews/article...,Post-IPO,50.0,United States,4/17/2026


In [2]:
df.shape

(4365, 11)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4365 entries, 0 to 4364
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   company              4365 non-null   object 
 1   location             4364 non-null   object 
 2   total_laid_off       2854 non-null   float64
 3   date                 4364 non-null   object 
 4   percentage_laid_off  2745 non-null   float64
 5   industry             4363 non-null   object 
 6   source               4362 non-null   object 
 7   stage                4360 non-null   object 
 8   funds_raised         3865 non-null   float64
 9   country              4363 non-null   object 
 10  date_added           4365 non-null   object 
dtypes: float64(3), object(8)
memory usage: 375.2+ KB


In [4]:
df.isnull().sum()

company                   0
location                  1
total_laid_off         1511
date                      1
percentage_laid_off    1620
industry                  2
source                    3
stage                     5
funds_raised            500
country                   2
date_added                0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
# Make column names easier to work with
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
0,Pentera,Boston,40.0,4/27/2026,0.08,Security,https://www.calcalistech.com/ctechnews/article...,Series D,249.0,United States,4/30/2026
1,SuperOps,"Chennai, Non-U.S.",60.0,4/24/2026,0.30,Other,https://inc42.com/buzz/superops-lays-off-30-st...,Series C,54.0,India,4/30/2026
2,Acko,"Mumbai, Non-U.S.",60.0,4/20/2026,0.05,Finance,https://inc42.com/buzz/acko-cuts-5-workforce-i...,Unknown,143.0,India,4/20/2026
3,Meta,SF Bay Area,8000.0,4/17/2026,0.10,Consumer,https://www.reuters.com/world/meta-targets-may...,Post-IPO,26000.0,United States,4/20/2026
4,Shutterfly,"Haifa, Non-U.S.",80.0,4/16/2026,NaN,Manufacturing,https://www.calcalistech.com/ctechnews/article...,Post-IPO,50.0,United States,4/17/2026


In [7]:
# Convert date column
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Remove duplicate rows
df = df.drop_duplicates()

# Clean text columns
text_cols = ["company", "location", "industry", "stage", "country"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4365 entries, 0 to 4364
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   company              4365 non-null   object        
 1   location             4365 non-null   object        
 2   total_laid_off       2854 non-null   float64       
 3   date                 4364 non-null   datetime64[ns]
 4   percentage_laid_off  2745 non-null   float64       
 5   industry             4365 non-null   object        
 6   source               4362 non-null   object        
 7   stage                4365 non-null   object        
 8   funds_raised         3865 non-null   float64       
 9   country              4365 non-null   object        
 10  date_added           4365 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 375.2+ KB


In [8]:
df.isnull().sum().sort_values(ascending=False)

percentage_laid_off    1620
total_laid_off         1511
funds_raised            500
source                    3
date                      1
company                   0
location                  0
industry                  0
stage                     0
country                   0
date_added                0
dtype: int64

In [9]:
# Fill missing numeric values with 0 for analysis
numeric_cols = ["total_laid_off", "percentage_laid_off", "funds_raised"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(0)

# Fill missing categories
category_cols = ["industry", "stage", "country", "location"]

for col in category_cols:
    if col in df.columns:
        df[col] = df[col].replace("nan", "Unknown")
        df[col] = df[col].fillna("Unknown")

In [10]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["year_month"] = df["date"].dt.to_period("M").astype(str)

df.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added,year,month,year_month
0,Pentera,Boston,40.0,2026-04-27,0.08,Security,https://www.calcalistech.com/ctechnews/article...,Series D,249.0,United States,4/30/2026,2026.0,4.0,2026-04
1,SuperOps,"Chennai, Non-U.S.",60.0,2026-04-24,0.30,Other,https://inc42.com/buzz/superops-lays-off-30-st...,Series C,54.0,India,4/30/2026,2026.0,4.0,2026-04
2,Acko,"Mumbai, Non-U.S.",60.0,2026-04-20,0.05,Finance,https://inc42.com/buzz/acko-cuts-5-workforce-i...,Unknown,143.0,India,4/20/2026,2026.0,4.0,2026-04
3,Meta,SF Bay Area,8000.0,2026-04-17,0.10,Consumer,https://www.reuters.com/world/meta-targets-may...,Post-IPO,26000.0,United States,4/20/2026,2026.0,4.0,2026-04
4,Shutterfly,"Haifa, Non-U.S.",80.0,2026-04-16,0.00,Manufacturing,https://www.calcalistech.com/ctechnews/article...,Post-IPO,50.0,United States,4/17/2026,2026.0,4.0,2026-04


In [11]:
df.to_csv("../data/cleaned/layoffs_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
